# 🐍 3 — Namespaces & Decorators

**Welcome!** 🎉 This session has two powerful topics. First, **Namespaces & Scope** — how Python decides *which* variable you mean when names repeat. Then **Decorators** — a clever way to add features to a function without changing its code.

Every topic is in **simple English**, with **real-life examples** and **line-by-line comments**.

---

## 📚 What you will learn here

| # | Topic |
|---|-------|
| 1 | Namespaces (the 4 types) |
| 2 | Scope & the **LEGB** rule |
| 3 | `local` vs `global` (and the `global` keyword) |
| 4 | Built-in scope |
| 5 | Enclosing scope & the `nonlocal` keyword |
| 6 | **Decorators** (functions that upgrade functions) |
| 7 | Practical decorators (timer, type-checker) |

---

# 🅰️ Part 1 — Namespaces & Scope

## What is a Namespace?

A **namespace** is a **space that holds names (identifiers)**. Technically, it's a **dictionary** where the keys are the variable names and the values are the objects they point to.

Example: if `a = 2` and `b = 3`, the namespace is `{a: 2, b: 3}`.

### The 4 types of namespaces
| Type | Where it lives |
|------|----------------|
| **Built-in** | names Python gives you for free (`print`, `len`, ...) |
| **Global** | names in your main program |
| **Enclosing** | names in an outer function (when functions are nested) |
| **Local** | names inside the current function |

## The LEGB Rule ⭐

A **scope** is the region of your program where a namespace is directly accessible.

When Python sees a name (like `a`), it searches for it **from the inside out**, in this order:

```
   L  →  E  →  G  →  B
 Local  Enclosing  Global  Built-in
```

If it isn't found in **any** of these, Python raises a `NameError`. This search order is called the **LEGB rule**. We'll see it in action below.

## 2️⃣ Local vs Global

- **Global scope** = your main program.
- **Local scope** = inside a function.

In [ ]:
a = 2            # global variable (in the main program's scope)

def temp():
    b = 3        # local variable (only exists inside this function)
    print(b)

temp()
print(a)

3
2


### Same name, different scopes
You *can* have two variables with the same name, because they live in **different namespaces** that don't affect each other.

In [ ]:
a = 2

def temp():
    a = 3        # a LOCAL 'a', separate from the global one
    print(a)

temp()
print(a)         # the global 'a' is unchanged

3
2


### Reading a global from inside a function
If a name isn't found locally, Python looks in the global scope (LEGB). So a function **can read** a global variable.

In [ ]:
a = 2

def temp():
    print(a)     # no local 'a', so Python finds the global 'a'

temp()
print(a)

2
2


### ⚠️ You can READ a global, but not CHANGE it directly
Trying to modify a global from inside a function (without permission) causes an error.

In [ ]:
a = 2

def temp():
    a += 1       # ❌ UnboundLocalError — can't change the global 'a' directly
    print(a)

temp()
print(a)

### The `global` keyword
To actually change a global from inside a function, declare it with **`global`**. *(Use this sparingly — it's usually better to return a value instead.)*

In [ ]:
a = 2

def temp():
    global a     # tell Python: I want to change the GLOBAL 'a'
    a += 1
    print(a)

temp()
print(a)         # the global 'a' really changed

3
3


You can even **create** a global variable from inside a function using `global`.

In [ ]:
def temp():
    global a
    a = 1        # this creates a GLOBAL 'a'
    print(a)

temp()
print(a)         # accessible outside too

1
1


### Function parameters are local
A function's parameter (like `z`) is always a **local** variable.

In [ ]:
def temp(z):
    print(z)     # z is local to temp

a = 5
temp(5)
print(a)

5
5


## 3️⃣ Built-in Scope

Have you ever wondered why `print()`, `len()`, `type()`, `max()` just work — without importing anything? They live in the **built-in scope**. Python loads them automatically for every program.

In [ ]:
print("hello")   # print lives in the built-in scope

hello


In [ ]:
# see ALL the built-in names
import builtins
print(dir(builtins))

['ArithmeticError', 'AssertionError', 'AttributeError', ..., 'print', 'range', 'set', 'str', 'sum', 'tuple', 'type', 'zip']


### ⚠️ Don't shadow built-ins!
Because of LEGB, a name you define in the **global** scope is found *before* the built-in one. So if you name your own function `max`, it **replaces** the real `max` — and things break.

In [ ]:
l = [1, 2, 3]
print(max(l))    # the real built-in max -> 3

def max():       # ❌ we just shadowed the built-in max!
    print("Hello")

print(max(l))    # ❌ TypeError: our max() takes 0 arguments but 1 was given

3


## 4️⃣ Enclosing Scope & `nonlocal`

When you put a **function inside a function**, the outer one's scope is called the **enclosing scope**.

In [ ]:
def outer():
    def inner():
        print("inner function")
    inner()
    print("outer function")

outer()
print("main program")

inner function
outer function
main program


**LEGB in a nested function:** the inner function first checks its own local scope. Here `a = 4` is local to `inner`, so it prints `4`.

In [ ]:
def outer():
    a = 3
    def inner():
        a = 4          # local to inner
        print(a)
    inner()
    print("outer function")

a = 1
outer()
print("main program")

4
outer function
main program


If `inner` has no `a`, LEGB looks in the **enclosing** scope (`outer`), finding `a = 3`.

In [ ]:
def outer():
    a = 3
    def inner():
        print(a)       # no local a -> look in enclosing (outer) -> 3
    inner()
    print("outer function")

a = 1
outer()
print("main program")

3
outer function
main program


And if neither has `a`, LEGB goes to the **global** scope, finding `a = 1`.

In [ ]:
def outer():
    def inner():
        print(a)       # not local, not enclosing -> global -> 1
    inner()
    print("outer function")

a = 1
outer()
print("main program")

1
outer function
main program


### The `nonlocal` keyword
Just like `global` changes a global variable, **`nonlocal`** lets an inner function **change a variable in the enclosing** function.

In [ ]:
def outer():
    a = 1
    def inner():
        nonlocal a     # change the enclosing 'a', not create a new local one
        a += 1
        print("inner", a)
    inner()
    print("outer", a)  # the enclosing 'a' really changed

outer()
print("main program")

inner 2
outer 2
main program


### 🧾 Scope summary
- **Global** → the main program.
- **Local** → inside a function.
- **Enclosing** → the outer of two nested functions.
- **Built-in** → always available, sits above everything.
- **LEGB** → search order: Local → Enclosing → Global → Built-in.

---

# 🅱️ Part 2 — Decorators

## What is a Decorator?

A **decorator** is a **function that takes another function as input, adds some extra feature ("decoration") to it, and returns it** — without changing the original function's code.

This is only possible because Python functions are **first-class citizens** — they are objects you can store, delete, pass as arguments, and return.

**Two kinds of decorators:**
- **Built-in:** `@staticmethod`, `@classmethod`, `@abstractmethod`, `@property`.
- **User-defined:** ones we create ourselves.

Decorators are written with the **`@`** symbol.

### First, proof that functions are first-class citizens

In [ ]:
# 1. store a function in a variable
def func():
    print("hello")

a = func         # 'a' now points to the same function
a()

hello


In [ ]:
# 2. delete a function
del func
func()           # ❌ NameError: name 'func' is not defined

In [ ]:
# 3. pass a function as an argument to another function
def modify(func, num):
    return func(num)

def square(num):
    return num ** 2

modify(square, 2)   # pass the 'square' function in -> 4

4

### A simple decorator
This decorator wraps a function's output with lines of `*`. The inner function is called **`wrapper`**.

In [ ]:
def my_decorator(func):          # takes a function as input
    def wrapper():               # an inner "wrapper" function
        print("********")
        func()                   # call the original function in the middle
        print("********")
    return wrapper               # return the wrapper

def hello():
    print("hello")

a = my_decorator(hello)          # decorate 'hello' -> returns the wrapper
a()

********
hello
********


The same decorator works on **any** function — reuse it freely.

In [ ]:
def my_decorator(func):
    def wrapper():
        print("********")
        func()
        print("********")
    return wrapper

def hello():
    print("hello")

def display():
    print("hello shahab")

my_decorator(hello)()
my_decorator(display)()

********
hello
********
********
hello shahab
********


### 🔒 Closures — the secret behind decorators
Normally, when a function finishes, all its local variables vanish. But if an **inner function** is returned, it **remembers** the variables of its parent function even after the parent is gone. This memory is called a **closure** — and it's exactly what makes decorators work.

### The clean `@` syntax
Writing `a = my_decorator(hello)` then `a()` is clunky. Python gives us a shortcut: put **`@decorator_name`** right above the function you want to decorate.

In [ ]:
def my_decorator(func):
    def wrapper():
        print("********")
        func()
        print("********")
    return wrapper

@my_decorator            # this one line does all the wrapping for us
def hello():
    print("hello")

hello()                  # already decorated!

********
hello
********


## 5️⃣ A Useful Decorator — a Timer ⏱️

Now something practical: a decorator that **measures how long a function takes to run**. This is a real technique used to find slow code.

In [ ]:
import time

def timer(func):
    def wrapper():
        start = time.time()
        func()
        print("time taken by", func.__name__, time.time() - start, "secs")
    return wrapper

@timer
def hello():
    print("hello world")
    time.sleep(2)

hello()

hello world
time taken by hello 2.0058975219726562 secs


The same timer works on multiple functions.

In [ ]:
import time

def timer(func):
    def wrapper():
        start = time.time()
        func()
        print("time taken by", func.__name__, time.time() - start, "secs")
    return wrapper

@timer
def hello():
    print("hello world")
    time.sleep(2)

@timer
def display():
    print("displaying something")
    time.sleep(4)

hello()
display()

hello world
time taken by hello 2.003 secs
displaying something
time taken by display 4.003 secs


### ⚠️ Problem: functions that take arguments
Our `wrapper()` takes no arguments, so it breaks on a function like `square(num)` that needs one.

In [ ]:
import time

def timer(func):
    def wrapper():            # takes no arguments
        start = time.time()
        func()
        print("time taken by", func.__name__, time.time() - start, "secs")
    return wrapper

@timer
def square(num):
    time.sleep(1)
    return num ** 2

square(2)   # ❌ TypeError: wrapper() takes 0 positional arguments but 1 was given

### ✅ The fix: `*args`
Make the wrapper accept **`*args`** (any number of arguments) and pass them along. Now the decorator works on **any** function, no matter how many inputs it takes.

In [ ]:
import time

def timer(func):
    def wrapper(*args):              # accept any number of arguments
        start = time.time()
        func(*args)                  # pass them to the original function
        print("time taken by", func.__name__, time.time() - start, "secs")
    return wrapper

@timer
def hello():
    print("hello world")
    time.sleep(2)

@timer
def square(num):
    time.sleep(1)
    print(num ** 2)

@timer
def power(a, b):
    print(a ** b)

hello()
square(2)
power(2, 3)

hello world
time taken by hello 2.003 secs
4
time taken by square 1.001 secs
8
time taken by power 3.5e-05 secs


## 6️⃣ A Decorator That Takes Arguments

Here's an advanced one: a decorator that **checks the data type** of a function's input before running it. This needs an **extra layer** — a function that returns a decorator.

**The problem it solves:** `square("hello")` crashes because you can't raise a string to a power. We want to *automatically* reject wrong types.

In [ ]:
# Without protection, wrong types crash the function:
def square(num):
    print(num ** 2)

square("hehehe")   # ❌ TypeError: can't do ** on a string

**The solution:** `sanity_check(data_type)` returns a decorator that only runs the function if the input matches the required type. Three layers: the outer takes the *type*, the middle takes the *function*, the inner takes the *arguments*.

> 🛠️ **Fixed from the original:** one cell used `@checkdata(int)` with an undefined `checkdata` (it was just a sketch of the goal). The working decorator is `sanity_check` below. Also cleaned the error message.

In [ ]:
def sanity_check(data_type):             # 1) takes the required type
    def outer_wrapper(func):             # 2) takes the function
        def inner_wrapper(*args):        # 3) takes the arguments
            if type(args[0]) == data_type:
                func(*args)              # type is correct -> run it
            else:
                raise TypeError("this data type is not allowed")
        return inner_wrapper
    return outer_wrapper

@sanity_check(int)
def square(num):
    print(num ** 2)

@sanity_check(str)
def greet(name):
    print("hello", name)

greet("shahab")   # correct type -> works
greet(5)          # ❌ wrong type -> raises our TypeError

hello shahab


---

## 🎯 Summary — Quick Revision

Two big ideas mastered! 🎉

| Concept | Key idea |
|---------|----------|
| **Namespace** | A dictionary of names → objects |
| **4 namespaces** | Built-in, Global, Enclosing, Local |
| **LEGB rule** | Search order: Local → Enclosing → Global → Built-in |
| **`global`** | Lets a function change a global variable |
| **`nonlocal`** | Lets an inner function change an enclosing variable |
| **First-class functions** | Functions can be stored, deleted, passed, returned |
| **Closure** | An inner function remembers its parent's variables |
| **Decorator** | A function that adds features to another function |
| **`@decorator`** | The clean shortcut syntax |
| **`*args` in wrapper** | Makes a decorator work on any function |

---

### 📝 Try It Yourself (Practice Questions)

1. Predict the output of a nested function before running it — practise the LEGB rule.
2. Write a decorator that prints `"Function started"` before and `"Function ended"` after a function runs.
3. Write a decorator that prints the **arguments** a function was called with.
4. Make a decorator that runs a function **twice**.
5. Extend `sanity_check` to check **all** arguments' types, not just the first one.

> 💪 **Remember:** decorators look magical at first, but they're just "a function wrapping another function." Read them from the inside out and they make perfect sense!

---

**➡️ Next up: 4 — Iterators & Generators.**